# Python Iterative Approch

In [0]:
# --- FUNCTION 1: Standardize Dates ---
def fix_dates(dataframe):
    for col in ["StartDate", "EndDate"]:
        dataframe[col] = pd.to_datetime(dataframe[col], format="mixed", dayfirst=True).dt.strftime("%Y-%m-%d")
    return dataframe

# --- FUNCTION 2: Merge Overlapping & Adjacent Windows ---
def merge_customer_windows(dataframe):
    merged_rows = []
    current_cust = None
    current_start = None
    current_end = None

    # Sort data by CustomerID and StartDate
    df_sorted = dataframe.sort_values(["CustomerID", "StartDate"])

    for _, row in df_sorted.iterrows():
        cust = row["CustomerID"]
        start = pd.to_datetime(row["StartDate"])
        end = pd.to_datetime(row["EndDate"])
        
        if current_cust != cust:
            # Save previous customer's active window before switching
            if current_cust is not None:
                merged_rows.append({
                    "CustomerID": current_cust,
                    "StartDate": current_start.strftime("%Y-%m-%d"),
                    "EndDate": current_end.strftime("%Y-%m-%d")
                })
            # Initialize for the new customer
            current_cust = cust
            current_start = start
            current_end = end
        else:
            # Check overlap or adjacency (current start <= previous end + 1 day)
            if start <= current_end + pd.Timedelta(days=1):
                current_end = max(current_end, end)
            else:
                # Save completed window and start a new one
                merged_rows.append({
                    "CustomerID": current_cust,
                    "StartDate": current_start.strftime("%Y-%m-%d"),
                    "EndDate": current_end.strftime("%Y-%m-%d")
                })
                current_start = start
                current_end = end

    # Append the very last processed window
    if current_cust is not None:
        merged_rows.append({
            "CustomerID": current_cust,
            "StartDate": current_start.strftime("%Y-%m-%d"),
            "EndDate": current_end.strftime("%Y-%m-%d")
        })

    return pd.DataFrame(merged_rows)


import pandas as pd

# Define the data
data = [
    {"CustomerID": "A001", "StartDate": "2023-01-01", "EndDate": "2023-01-10"},
    {"CustomerID": "A001", "StartDate": "2023-01-08", "EndDate": "2023-01-15"},
    {"CustomerID": "A001", "StartDate": "2023-01-20", "EndDate": "2023-01-25"},
    {"CustomerID": "A001", "StartDate": "2023-01-25", "EndDate": "2023-01-30"},
    {"CustomerID": "A001", "StartDate": "2023-02-05", "EndDate": "2023-02-10"},
    {"CustomerID": "A002", "StartDate": "01/01/2025", "EndDate": "31/10/2025"},
    {"CustomerID": "A002", "StartDate": "11/04/2025", "EndDate": "23/11/2025"},
    {"CustomerID": "A002", "StartDate": "15/05/2024", "EndDate": "02/11/2024"},
]

df = pd.DataFrame(data)
df = fix_dates(df)
df_final = merge_customer_windows(df)
df_final




,CustomerID,StartDate,EndDate
0,A001,2023-01-01,2023-01-15
1,A001,2023-01-20,2023-01-30
2,A001,2023-02-05,2023-02-10
3,A002,2024-05-15,2024-11-02
4,A002,2025-01-01,2025-11-23


# Python Recursive Approch

In [0]:
# 1. Standardize dates function
def fix_dates(dataframe):
    for col in ["StartDate", "EndDate"]:
        dataframe[col] = pd.to_datetime(dataframe[col], format="mixed", dayfirst=True).dt.strftime("%Y-%m-%d")
    return dataframe


# 2. Recursive function to merge intervals
def recursive_merge_intervals(intervals):
    # Base case: if 0 or 1 interval remains, it's already merged
    if len(intervals) <= 1:
        return intervals
    
    curr_start, curr_end = intervals[0]
    next_start, next_end = intervals[1]
    
    # Check overlap or adjacency (next start <= current end + 1 day)
    if next_start <= curr_end + pd.Timedelta(days=1):
        # Merge the first two intervals and recurse with the combined interval at the front
        merged = (curr_start, max(curr_end, next_end))
        return recursive_merge_intervals([merged] + intervals[2:])
    else:
        # No overlap/adjacency; keep the first interval and recurse on the rest
        return [intervals[0]] + recursive_merge_intervals(intervals[1:])
    

# 3. Main processing function using recursion per customer
def process_customers_recursively(dataframe):
    df_sorted = dataframe.sort_values(["CustomerID", "StartDate"])
    final_rows = []
    
    # Group by customer so recursion happens independently per CustomerID
    for cust, group in df_sorted.groupby("CustomerID"):
        # Convert row dates to a list of (start, end) datetime tuples
        intervals = list(zip(
            pd.to_datetime(group["StartDate"]), 
            pd.to_datetime(group["EndDate"])
        ))
        
        # Apply the recursive merge function
        merged_intervals = recursive_merge_intervals(intervals)
        
        # Format back into dictionary rows
        for start, end in merged_intervals:
            final_rows.append({
                "CustomerID": cust,
                "StartDate": start.strftime("%Y-%m-%d"),
                "EndDate": end.strftime("%Y-%m-%d")
            })
            
    return pd.DataFrame(final_rows)

import pandas as pd

# Define the data
data = [
    {"CustomerID": "A001", "StartDate": "2023-01-01", "EndDate": "2023-01-10"},
    {"CustomerID": "A001", "StartDate": "2023-01-08", "EndDate": "2023-01-15"},
    {"CustomerID": "A001", "StartDate": "2023-01-20", "EndDate": "2023-01-25"},
    {"CustomerID": "A001", "StartDate": "2023-01-25", "EndDate": "2023-01-30"},
    {"CustomerID": "A001", "StartDate": "2023-02-05", "EndDate": "2023-02-10"},
    {"CustomerID": "A002", "StartDate": "01/01/2025", "EndDate": "31/10/2025"},
    {"CustomerID": "A002", "StartDate": "11/04/2025", "EndDate": "23/11/2025"},
    {"CustomerID": "A002", "StartDate": "15/05/2024", "EndDate": "02/11/2024"},
]

df = pd.DataFrame(data)

# --- EXECUTION ---
df = fix_dates(df)
df_final = process_customers_recursively(df)
df_final

,CustomerID,StartDate,EndDate
0,A001,2023-01-01,2023-01-15
1,A001,2023-01-20,2023-01-30
2,A001,2023-02-05,2023-02-10
3,A002,2024-05-15,2024-11-02
4,A002,2025-01-01,2025-11-23


#Python OOP 

In [0]:
import pandas as pd

class DateIntervalProcessor:
    def __init__(self, data):
        """Initialize the class with raw data."""
        self.df = pd.DataFrame(data)

    def fix_dates(self):
        """Standardize all date columns into uniform 'YYYY-MM-DD' strings."""
        for col in ["StartDate", "EndDate"]:
            self.df[col] = pd.to_datetime(self.df[col], format="mixed", dayfirst=True).dt.strftime("%Y-%m-%d")
        return self

    def merge_customer_windows(self):
        """Iteratively sort and merge overlapping/adjacent date windows per customer."""
        merged_rows = []
        current_cust = None
        current_start = None
        current_end = None

        # Sort data chronologically by CustomerID and StartDate
        df_sorted = self.df.sort_values(["CustomerID", "StartDate"])

        # Iterative loop over rows
        for _, row in df_sorted.iterrows():
            cust = row["CustomerID"]
            start = pd.to_datetime(row["StartDate"])
            end = pd.to_datetime(row["EndDate"])
            
            if current_cust != cust:
                # Save previous customer's active window before switching
                if current_cust is not None:
                    merged_rows.append({
                        "CustomerID": current_cust,
                        "StartDate": current_start.strftime("%Y-%m-%d"),
                        "EndDate": current_end.strftime("%Y-%m-%d")
                    })
                # Initialize for the new customer
                current_cust = cust
                current_start = start
                current_end = end
            else:
                # Check overlap or adjacency (current start <= previous end + 1 day)
                if start <= current_end + pd.Timedelta(days=1):
                    current_end = max(current_end, end)
                else:
                    # Save completed window and start a new one
                    merged_rows.append({
                        "CustomerID": current_cust,
                        "StartDate": current_start.strftime("%Y-%m-%d"),
                        "EndDate": current_end.strftime("%Y-%m-%d")
                    })
                    current_start = start
                    current_end = end

        # Append the final processed window
        if current_cust is not None:
            merged_rows.append({
                "CustomerID": current_cust,
                "StartDate": current_start.strftime("%Y-%m-%d"),
                "EndDate": current_end.strftime("%Y-%m-%d")
            })

        # Update the object's dataframe with the final result
        self.df = pd.DataFrame(merged_rows)
        return self.df


# --- EXECUTION ---
data = [
    {"CustomerID": "A001", "StartDate": "2023-01-01", "EndDate": "2023-01-10"},
    {"CustomerID": "A001", "StartDate": "2023-01-08", "EndDate": "2023-01-15"},
    {"CustomerID": "A001", "StartDate": "2023-01-20", "EndDate": "2023-01-25"},
    {"CustomerID": "A001", "StartDate": "2023-01-25", "EndDate": "2023-01-30"},
    {"CustomerID": "A001", "StartDate": "2023-02-05", "EndDate": "2023-02-10"},
    {"CustomerID": "A002", "StartDate": "01/01/2025", "EndDate": "31/10/2025"},
    {"CustomerID": "A002", "StartDate": "11/04/2025", "EndDate": "23/11/2025"},
    {"CustomerID": "A002", "StartDate": "15/05/2024", "EndDate": "02/11/2024"},
]

# Create object, run methods sequentially
processor = DateIntervalProcessor(data)
processor.fix_dates()
df_final = processor.merge_customer_windows()

df_final

,CustomerID,StartDate,EndDate
0,A001,2023-01-01,2023-01-15
1,A001,2023-01-20,2023-01-30
2,A001,2023-02-05,2023-02-10
3,A002,2024-05-15,2024-11-02
4,A002,2025-01-01,2025-11-23


#SQL

In [0]:
%sql
DROP TABLE IF EXISTS Trans_Category;

-- Then drop the parent table
DROP TABLE IF EXISTS Account;

In [0]:
%sql
-- 1. Create the Account table
CREATE TABLE Account (
    Account INT PRIMARY KEY,
    Open_Date DATE NOT NULL,
    Close_Date DATE NOT NULL
);

-- 2. Create the Trans_Category table
CREATE TABLE Trans_Category (
    Transaction_Key INT PRIMARY KEY,
    Account INT NOT NULL,
    Category VARCHAR(50) NOT NULL,
    FOREIGN KEY (Account) REFERENCES Account(Account)
);

-- 3. Insert data into the Account table
INSERT INTO Account (Account, Open_Date, Close_Date) VALUES
(534354324, '2010-02-01', '3999-12-31'),
(6543254,   '2013-04-23', '3999-12-31'),
(673524,    '2015-12-15', '3999-12-31'),
(9786536,   '2012-05-23', '3999-12-31'),
(12554855,  '2014-03-24', '2016-01-01');

-- 4. Insert data into the Trans_Category table
INSERT INTO Trans_Category (Account, Transaction_Key, Category) VALUES
(534354324, 1,  'Toys'),
(534354324, 2,  'Beds'),
(534354324, 3,  'TV'),
(6543254,   4,  'HiFi'),
(673524,    5,  'Sport'),
(673524,    6,  'TV'),
(673524,    7,  'Beds'),
(673524,    8,  'Toys'),
(673524,    9,  'TV'),
(9786536,   10, 'TV'),
(12554855,  11, 'Toys'),
(12554855,  12, 'HiFi'),
(12554855,  13, 'iPod');

num_affected_rows,num_inserted_rows
13,13


In [0]:
%sql
SELECT DISTINCT Account.Account, Account.Open_Date, Account.Close_Date
FROM Account 
JOIN Trans_Category  ON Account.Account = Trans_Category.Account
WHERE Account.Close_Date = '3999-12-31'AND Trans_Category.Category = 'TV';

Account,Open_Date,Close_Date
9786536,2012-05-23,3999-12-31
673524,2015-12-15,3999-12-31
534354324,2010-02-01,3999-12-31


In [0]:
%sql
SELECT Account, Open_Date, Close_Date
FROM Account
WHERE Close_Date = '3999-12-31' AND Account NOT IN (
      SELECT Account 
      FROM Trans_Category 
      WHERE Category = 'Beds'
  );

Account,Open_Date,Close_Date
6543254,2013-04-23,3999-12-31
9786536,2012-05-23,3999-12-31
